# Groovy kernel on a hosted Colab runtime (bootstrap)

This notebook's metadata names the `groovy` kernelspec. Colab can't find it yet, so it opens
with the Python runtime — that's expected. **Pick your hardware first** (Runtime → Change runtime
type → T4 GPU) because changing the accelerator later resets the VM and loses the install.

1. Run the two Python cells below (build + install the kernel, then a transport pre-check).
2. **Reload the browser tab** (keep the runtime connected). Colab re-reads the kernelspec named in
   this notebook and should now connect to the Groovy kernel.
3. Run the Groovy cell at the bottom.


In [ ]:
%%bash
set -e
java -version 2>&1 | head -1
echo "COLAB_JUPYTER_TRANSPORT=${COLAB_JUPYTER_TRANSPORT:-<unset>}"
cd /content
[ -d groovy-jupyter ] || git clone -q --depth 1 https://github.com/paulk-asert/groovy-jupyter.git
cd groovy-jupyter
./gradlew --no-daemon -q kernelSpecZip
rm -rf /content/kspec && mkdir -p /content/kspec
unzip -q build/distributions/groovy-jupyter-kernelspec-*.zip -d /content/kspec
jupyter kernelspec install /content/kspec --name groovy --user
jupyter kernelspec list


In [ ]:
# Pre-check without touching the UI: start the Groovy kernel exactly as Colab's launcher would
# (transport=ipc) and ask it to execute something. Expect "ipc OK: ok".
from jupyter_client import KernelManager
km = KernelManager(kernel_name="groovy")
km.transport = "ipc"
km.ip = "/tmp/groovy-ipc-check"
km.start_kernel()
kc = km.client(); kc.start_channels()
try:
    kc.wait_for_ready(timeout=90)
    r = kc.execute("GroovySystem.version", reply=True, timeout=30)
    print("ipc OK:", r["content"]["status"])
except Exception as e:
    print("ipc FAILED:", e)
finally:
    kc.stop_channels(); km.shutdown_kernel(now=True)


**Now reload the browser tab.** The status area should say the runtime is connected and the
kernel picker (Runtime → Change runtime type) should show *Groovy 6.0.0-beta-3*. If Colab did not
switch automatically, select it there. Then run the next cell — it is Groovy, so it only works once
the Groovy kernel is live.

In [ ]:
println "Groovy ${GroovySystem.version} on Java ${System.getProperty('java.version')}"
// the connection file Colab wrote for this kernel — proves which transport the hosted launcher used
new File('/root/.local/share/jupyter/runtime').listFiles()
    .findAll { it.name.startsWith('kernel-') && it.name.endsWith('.json') }
    .each { println it.text }


In [ ]:
// GPU sanity (T4 runtime): the CUDA driver/toolkit are on the image; JCuda arrives via @Grab
['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'].execute().text